In [51]:
import ast
from collections import deque

def load_environment(filepath):
    with open(filepath, 'r') as file:
        # Khali lines ko ignore karke data read karein
        lines = [line.strip() for line in file if line.strip()]
    
    # 1. Grid Dimensions
    rows, cols = map(int, lines[0].split(','))
    grid = [['.' for _ in range(cols)] for _ in range(rows)]
    
    # 2. Obstacles
    num_obstacles = int(lines[1])
    if num_obstacles > 0:
        # ast.literal_eval string ko tuple list mein convert karta hai e.g., "(1,3), (2,3)" -> [(1,3), (2,3)]
        obstacles = ast.literal_eval(f"[{lines[2]}]")
        for r, c in obstacles:
            grid[r][c] = 'X'
            
    # 3. Dirty Locations
    num_dirty = int(lines[3])
    dirty_locations = []
    if num_dirty > 0:
        dirty_locations = ast.literal_eval(f"[{lines[4]}]")
        for i, (r, c) in enumerate(dirty_locations):
            grid[r][c] = f'D{i+1}' # D1, D2, D3, D4
            
    # 4. Charging Station
    cs_tuple = ast.literal_eval(lines[5])
    if isinstance(cs_tuple[0], int): # Agar sirf ek tuple hai
         cs_r, cs_c = cs_tuple
    else:
         cs_r, cs_c = cs_tuple[0] # Fallback
    grid[cs_r][cs_c] = 'C'
    
    return grid, (cs_r, cs_c), dirty_locations

# --- TESTING THE LOADER ---
print("--- Loading Environment.txt ---")
my_grid, my_cs, my_dirty = load_environment('Environment.txt')

print(f"Charging Station: {my_cs}")
print(f"Dirty Locations: {my_dirty}")
print("\nGrid:")
for row in my_grid:
    print("  ".join(row))

--- Loading Environment.txt ---
Charging Station: (3, 0)
Dirty Locations: [(2, 4), (5, 6), (4, 1), (6, 3)]

Grid:
.  .  .  .  .  .  .  .
.  .  .  X  .  .  X  .
.  .  .  .  D1  .  .  .
C  .  .  .  .  .  .  .
.  D3  .  .  X  .  .  .
.  .  X  .  .  .  D2  .
X  .  .  D4  .  X  .  .
.  .  .  .  .  .  .  .


In [52]:
def get_successors(state, grid):
    """Returns valid (next_state, action, step_cost) from current state."""
    r, c = state
    rows = len(grid)
    cols = len(grid[0])
    
    # UP, DOWN, LEFT, RIGHT
    moves = [(-1, 0, 'UP'), (1, 0, 'DOWN'), (0, -1, 'LEFT'), (0, 1, 'RIGHT')]
    successors = []
    
    for dr, dc, action in moves:
        nr, nc = r + dr, c + dc
        # Check boundaries and obstacles
        if 0 <= nr < rows and 0 <= nc < cols:
            if grid[nr][nc] != 'X':
                successors.append(((nr, nc), action, 1))
                
    return successors

def goal_test(state, goal_state):
    """Checks if robot reached the goal."""
    return state == goal_state

def path_cost(path):
    """Calculates total cost. Cost = number of moves (states - 1)."""
    if not path:
        return 0
    return len(path) - 1

In [53]:
def bfs_search(grid, start, goal):
    if goal_test(start, goal):
        return [start], 0, 0 # Path, Cost, Nodes Expanded
        
    frontier = deque([start]) # Queue for BFS
    came_from = {start: None} # To reconstruct path
    nodes_expanded = 0
    
    while frontier:
        current = frontier.popleft()
        nodes_expanded += 1
        
        if goal_test(current, goal):
            # Path reconstruct karein
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            path.reverse()
            return path, path_cost(path), nodes_expanded
            
        # Successors nikalein
        for next_state, action, step_cost in get_successors(current, grid):
            if next_state not in came_from:
                frontier.append(next_state)
                came_from[next_state] = current
                
    return None, 0, nodes_expanded # No path found

def search(grid, start, goal, algorithm="BFS"):
    if algorithm.upper() == "BFS":
        return bfs_search(grid, start, goal)
    elif algorithm.upper() == "A*":
        return a_star_search(grid, start, goal) # <-- P2 will add this line
    else:
        return None, 0, 0

def validate_path(path, grid):
    """Check karta hai ke path valid hai ya nahi (no diagonals, no obstacles)."""
    if not path: return True
    for i in range(len(path) - 1):
        r1, c1 = path[i]
        r2, c2 = path[i+1]
        # Adjacent check (Manhattan distance == 1)
        if abs(r1 - r2) + abs(c1 - c2) != 1:
            print(f"Error: Invalid move between {path[i]} and {path[i+1]}")
            return False
        # Obstacle check
        if grid[r2][c2] == 'X':
            print(f"Error: Path goes through obstacle at {path[i+1]}")
            return False
    return True

In [54]:
# Test 1: Sample Environment
print("--- Testing BFS on Environment_Sample.txt ---")
s_grid, s_cs, s_dirty = load_environment('Environment_Sample.txt')

start_pos = s_cs
goal_pos = s_dirty[0] # D1 is at (0,7)

print(f"Start: {start_pos} -> Goal: {goal_pos} (D1)")

path, cost, nodes = search(s_grid, start_pos, goal_pos, algorithm="BFS")

print(f"\nPath found: {path}")
print(f"Path Cost: {cost}")
print(f"Nodes Expanded: {nodes}")

# Path validity check
is_valid = validate_path(path, s_grid)
print(f"Is Path Valid? {is_valid}")

--- Testing BFS on Environment_Sample.txt ---
Start: (0, 0) -> Goal: (0, 7) (D1)

Path found: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7)]
Path Cost: 7
Nodes Expanded: 32
Is Path Valid? True


In [55]:
print("\n--- Testing BFS on YOUR Environment.txt ---")
# Ye grid upar Cell 1 mein load ho chuki hai (my_grid)
start_pos = my_cs
goal_pos = my_dirty[0] # D1 at (2,4)

print(f"Start: {start_pos} -> Goal: {goal_pos} (D1)")

path, cost, nodes = search(my_grid, start_pos, goal_pos, algorithm="BFS")

print(f"\nPath found: {path}")
print(f"Path Cost: {cost}")
print(f"Nodes Expanded: {nodes}")

is_valid = validate_path(path, my_grid)
print(f"Is Path Valid? {is_valid}")


--- Testing BFS on YOUR Environment.txt ---
Start: (3, 0) -> Goal: (2, 4) (D1)

Path found: [(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4)]
Path Cost: 5
Nodes Expanded: 23
Is Path Valid? True


In [56]:
import heapq

def heuristic(state, goal_state):
    # Manhattan Distance (Handout requirement)
    return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])

def a_star_search(grid, start, goal):
    if goal_test(start, goal):
        return [start], 0, 0
        
    # Priority Queue: (f_cost, g_cost, current_state)
    frontier = []
    heapq.heappush(frontier, (0, 0, start))
    
    came_from = {start: None}
    g_costs = {start: 0} # g(n) = cost from start to node
    nodes_expanded = 0
    
    while frontier:
        f_cost, current_g, current = heapq.heappop(frontier)
        nodes_expanded += 1
        
        if goal_test(current, goal):
            # Reconstruct path
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            path.reverse()
            return path, path_cost(path), nodes_expanded
            
        for next_state, action, step_cost in get_successors(current, grid):
            new_g = current_g + step_cost
            
            # Agar naya rasta chhota hai ya pehli baar dekh rahe hain
            if next_state not in g_costs or new_g < g_costs[next_state]:
                g_costs[next_state] = new_g
                f_cost = new_g + heuristic(next_state, goal)
                heapq.heappush(frontier, (f_cost, new_g, next_state))
                came_from[next_state] = current
                
    return None, 0, nodes_expanded # No path found

In [57]:
print("--- Algorithm Comparison (Start to D1) ---")
start = (3, 0) # Charging Station
goal = (2, 4)  # D1

# BFS Run
bfs_path, bfs_cost, bfs_nodes = search(my_grid, start, goal, algorithm="BFS")
# A* Run
astar_path, astar_cost, astar_nodes = search(my_grid, start, goal, algorithm="A*")

print(f"BFS  -> Cost: {bfs_cost}, Nodes Expanded: {bfs_nodes}")
print(f"A*   -> Cost: {astar_cost}, Nodes Expanded: {astar_nodes}")

--- Algorithm Comparison (Start to D1) ---
BFS  -> Cost: 5, Nodes Expanded: 23
A*   -> Cost: 5, Nodes Expanded: 10


In [58]:
print("--- Testing FAILURE CASE (Environment_Error.txt) ---")
# Error environment load karein
err_grid, err_cs, err_dirty = load_environment('Environment_Error.txt')

print("Grid Visualization (D1 is surrounded by X):")
for row in err_grid:
    print("  ".join(row))

# Start aur Goal set karein (D1 is blocked)
start_pos = err_cs
goal_pos = err_dirty[0] # D1 at (2,4)

print(f"\nStart: {start_pos} -> Goal: {goal_pos} (D1)")
print("Trying to find path using BFS...")

# Search function call karein
path, cost, nodes = search(err_grid, start_pos, goal_pos, algorithm="BFS")

# Check karein ke path mila ya nahi
if path is None:
    print("\n[FAILURE DETECTED]")
    print(f"Result: No path exists from {start_pos} to {goal_pos}.")
    print("The robot cannot reach the dirty location because it is surrounded by obstacles.")
    print(f"Nodes expanded before giving up: {nodes}")
else:
    print(f"\n[SUCCESS] Path found: {path}")
    print(f"Path Cost: {cost}")

--- Testing FAILURE CASE (Environment_Error.txt) ---
Grid Visualization (D1 is surrounded by X):
.  .  .  .  .  .  .  .
.  .  .  X  X  .  X  .
.  .  .  X  D1  X  .  .
C  .  .  .  X  .  .  .
.  D3  .  .  X  .  .  .
.  .  X  .  .  .  D2  .
X  .  .  D4  .  X  .  .
.  .  .  .  .  .  .  .

Start: (3, 0) -> Goal: (2, 4) (D1)
Trying to find path using BFS...

[FAILURE DETECTED]
Result: No path exists from (3, 0) to (2, 4).
The robot cannot reach the dirty location because it is surrounded by obstacles.
Nodes expanded before giving up: 53


In [59]:
def load_schedule(filepath):
    with open(filepath, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    num_slots = int(lines[0])
    slots = [f"T{i+1}" for i in range(num_slots)]
    
    num_prec = int(lines[1])
    precedence = []
    for i in range(2, 2 + num_prec):
        # "(D1,D3)" ko ['D1', 'D3'] mein todna
        u, v = lines[i].replace('(', '').replace(')', '').split(',')
        precedence.append((u.strip(), v.strip()))
        
    latest_idx = 2 + num_prec
    num_latest = int(lines[latest_idx])
    latest_slot = {}
    for i in range(latest_idx + 1, latest_idx + 1 + num_latest):
        task, slot = lines[i].replace('(', '').replace(')', '').split(',')
        latest_slot[task.strip()] = slot.strip()
        
    return slots, precedence, latest_slot

def slot_to_int(slot):
    return int(slot[1:]) # "T3" ko 3 banayega

print("Schedule Loader Ready!")

Schedule Loader Ready!


In [60]:
import copy

def is_consistent(variable, value, assignment, precedence, latest_slot):
    # 1. Slot Exclusivity (Ek time slot mein sirf ek task)
    if value in assignment.values():
        return False
    
    # 2. Latest Slot Constraint
    if variable in latest_slot:
        if slot_to_int(value) > slot_to_int(latest_slot[variable]):
            return False
            
    # 3. Precedence Constraint
    for u, v in precedence:
        # Agar hum 'v' assign kar rahe hain, to 'u' pehle se assigned hona chahiye aur uski value choti honi chahiye
        if v == variable and u in assignment:
            if slot_to_int(assignment[u]) >= slot_to_int(value):
                return False
        # Agar hum 'u' assign kar rahe hain, to 'v' agar pehle se assigned hai, to uski value bari honi chahiye
        if u == variable and v in assignment:
            if slot_to_int(value) >= slot_to_int(assignment[v]):
                return False
    return True

def forward_check(variable, value, assignment, domains, precedence, latest_slot):
    new_domains = copy.deepcopy(domains)
    for unassigned_var in domains:
        if unassigned_var not in assignment:
            # Slot exclusivity
            if value in new_domains[unassigned_var]:
                new_domains[unassigned_var].remove(value)
            
            # Latest slot
            if unassigned_var in latest_slot:
                new_domains[unassigned_var] = [s for s in new_domains[unassigned_var] 
                                              if slot_to_int(s) <= slot_to_int(latest_slot[unassigned_var])]
            
            # Precedence
            for u, v in precedence:
                if v == unassigned_var and variable == u:
                    new_domains[unassigned_var] = [s for s in new_domains[unassigned_var] 
                                                  if slot_to_int(s) > slot_to_int(value)]
                if u == unassigned_var and variable == v:
                    new_domains[unassigned_var] = [s for s in new_domains[unassigned_var] 
                                                  if slot_to_int(s) < slot_to_int(value)]
            
            if not new_domains[unassigned_var]: # Agar domain khali ho jaye
                return None
    return new_domains

In [61]:
def select_unassigned_variable(assignment, domains, precedence):
    unassigned = [v for v in domains if v not in assignment]
    if not unassigned: return None
    
    # MRV (Minimum Remaining Values)
    min_domain_len = min(len(domains[v]) for v in unassigned)
    candidates = [v for v in unassigned if len(domains[v]) == min_domain_len]
    
    # Degree Heuristic (Tie breaker)
    if len(candidates) > 1:
        def degree(var):
            count = 0
            for u, v in precedence:
                if u == var or v == var:
                    count += 1
            return count
        return max(candidates, key=degree)
    return candidates[0]

def order_domain_values(variable, assignment, domains, precedence, latest_slot):
    # LCV (Least Constraining Value) - optional but implemented as per requirement
    if len(domains[variable]) == 1:
        return domains[variable]
    return domains[variable] # Sorting LCV ke liye thoda complex ho jata hai, filhal basic order return kar rahe hain

def backtracking_search(variables, domains, precedence, latest_slot):
    def backtrack(assignment, current_domains):
        if len(assignment) == len(variables):
            return assignment
            
        var = select_unassigned_variable(assignment, current_domains, precedence)
        for value in order_domain_values(var, assignment, current_domains, precedence, latest_slot):
            if is_consistent(var, value, assignment, precedence, latest_slot):
                assignment[var] = value
                new_domains = forward_check(var, value, assignment, current_domains, precedence, latest_slot)
                
                if new_domains is not None:
                    result = backtrack(assignment, new_domains)
                    if result is not None:
                        return result
                
                # Backtrack
                del assignment[var]
        return None
        
    return backtrack({}, domains)

In [62]:
print("--- Testing CSP on Schedule.txt ---")
slots, prec, latest = load_schedule('Schedule.txt')
# Variables Environment.txt se aayenge: D1, D2, D3, D4
variables = ['D1', 'D2', 'D3', 'D4']
domains = {v: slots.copy() for v in variables}

solution = backtracking_search(variables, domains, prec, latest)
print("Generated Schedule:", solution)

--- Testing CSP on Schedule.txt ---
Generated Schedule: {'D1': 'T2', 'D3': 'T1', 'D2': 'T3', 'D4': 'T4'}


In [63]:
print("\n--- Testing CSP on Schedule_Sample.txt ---")
slots_s, prec_s, latest_s = load_schedule('Schedule_Sample.txt')
domains_s = {v: slots_s.copy() for v in variables}
solution_s = backtracking_search(variables, domains_s, prec_s, latest_s)
print("Generated Schedule:", solution_s)


--- Testing CSP on Schedule_Sample.txt ---
Generated Schedule: {'D1': 'T1', 'D2': 'T2', 'D3': 'T3', 'D4': 'T4'}


In [64]:
print("\n--- Testing FAILURE CASE on Schedule_Error.txt ---")
slots_e, prec_e, latest_e = load_schedule('Schedule_Error.txt')
domains_e = {v: slots_e.copy() for v in variables}
solution_e = backtracking_search(variables, domains_e, prec_e, latest_e)

if solution_e is None:
    print("[FAILURE DETECTED] No feasible schedule exists due to contradictory constraints.")
else:
    print("Generated Schedule:", solution_e)


--- Testing FAILURE CASE on Schedule_Error.txt ---
[FAILURE DETECTED] No feasible schedule exists due to contradictory constraints.


In [65]:
def execute_schedule(grid, charging_station, schedule, algorithm):
    """
    Executes the CSP-generated schedule using graph-based search.

    grid             : environment loaded from Environment.txt
    charging_station : (row, col) of the charger
    schedule         : cleaning tasks in CSP slot order, e.g.
                       [("D3", (4, 1)), ("D1", (2, 4)), ("D2", (5, 6)), ("D4", (6, 3))]
    algorithm        : "BFS" or "A*"

    Returns: (complete_path, total_cost, success)
    """
    current = charging_station                 # 1. robot starts at the charger
    complete_path = [charging_station]
    cleaned = []
    leg_costs = []

    # 2-3. cleaning locations in scheduled order, then the return trip to the charger
    targets = list(schedule) + [("Charger", charging_station)]

    for label, goal in targets:
        path, cost, nodes = search(grid, current, goal, algorithm)

        if path is None:                       # 9. required path not found -> report failure
            print(f"[FAILURE] No path from {current} to {label} {goal}.")
            print(f"Cleaned so far: {cleaned if cleaned else 'none'}")
            return complete_path, path_cost(complete_path), False

        complete_path.extend(path[1:])         # 4. skip the duplicated start cell
        leg_costs.append(cost)
        current = goal                         # 5. update position
        if label != "Charger":
            cleaned.append(label)              #    mark dirty location as cleaned

    total_cost = path_cost(complete_path)      # 7. total navigation cost
    assert total_cost == sum(leg_costs)        # sanity check: legs add up to the full path

    return complete_path, total_cost, True     # 8. success

In [66]:
# 1. Environment load karein
grid, cs, dirty = load_environment('Environment.txt')
dirty_map = {f"D{i+1}": d for i, d in enumerate(dirty)}

# 2. CSP Solution ko slot ke hisaab se sort karein (YAHAN GHALTI THI)
# 'slot_num' ki jagah 'slot_to_int' likhein
order = sorted(solution, key=lambda t: slot_to_int(solution[t]))
schedule = [(t, dirty_map[t]) for t in order]

# 3. Thoda sa print karke confirm karein
print("CSP Generated Schedule:", solution)
print("Cleaning Order (Task, Coordinates):", schedule)
print("\n--- Starting Robot Simulation ---")

# 4. Integration Function ko call karein
path, cost, ok = execute_schedule(grid, cs, schedule, "A*")

# 5. Final Output Print karein
print("\n==========================================")
print("FINAL SIMULATION RESULTS:")
print("==========================================")
print(f"Robot complete Path: {path}")
print(f"Total Movements (Cost): {cost}")
print(f"Was Cleaning Successful & Returned?: {ok}")

# Final Grid dikhane ke liye (Dirty locations ab '.' ban jayengi)
print("\nFinal Grid (After Cleaning):")
final_grid = [row[:] for row in grid]
for loc in dirty:
    final_grid[loc[0]][loc[1]] = '.' # D1, D2 waghera ko clean ('.') kar dein
for row in final_grid:
    print("  ".join(row))

CSP Generated Schedule: {'D1': 'T2', 'D3': 'T1', 'D2': 'T3', 'D4': 'T4'}
Cleaning Order (Task, Coordinates): [('D3', (4, 1)), ('D1', (2, 4)), ('D2', (5, 6)), ('D4', (6, 3))]

--- Starting Robot Simulation ---

FINAL SIMULATION RESULTS:
Robot complete Path: [(3, 0), (3, 1), (4, 1), (3, 1), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (3, 6), (4, 6), (5, 6), (5, 5), (5, 4), (5, 3), (6, 3), (5, 3), (4, 3), (3, 3), (3, 2), (3, 1), (3, 0)]
Total Movements (Cost): 22
Was Cleaning Successful & Returned?: True

Final Grid (After Cleaning):
.  .  .  .  .  .  .  .
.  .  .  X  .  .  X  .
.  .  .  .  .  .  .  .
C  .  .  .  .  .  .  .
.  .  .  .  X  .  .  .
.  .  X  .  .  .  .  .
X  .  .  .  .  X  .  .
.  .  .  .  .  .  .  .
